# 01-01 Tensor 基础：创建、dtype 与 NumPy 互转

这一份只讲 Tensor 是什么、怎么创建、怎么看属性、怎么做类型转换，以及 Tensor 和 NumPy 怎么互转。


In [ ]:
import torch
import numpy as np

torch.manual_seed(42)
print("torch version:", torch.__version__)


## 0. 先说人话：Tensor 到底是什么

Tensor 就是 PyTorch 里的“多维数值容器”。

你可以先这样理解：

| 数学/机器学习里的东西 | PyTorch 里的形状 | 例子 |
|---|---:|---|
| 一个数，标量 | `[]` | loss = 0.35 |
| 一个向量 | `[特征数]` | 一个样本有 4 个特征 |
| 一个矩阵 | `[行, 列]` | 一批表格数据 |
| 一批图片 | `[batch, channel, height, width]` | 32 张 RGB 图片：`[32, 3, 224, 224]` |

PyTorch 里的模型、输入、标签、权重、梯度，基本都用 Tensor 表示。

## 1. 环境检查

推荐运行环境：`D:\\PythonWorkSpace\\anaconda\\envs\\pytorch\\python.exe`。

当前机器上这套环境可以导入 PyTorch。由于你的显卡比较新，当前 PyTorch 的 CUDA 版本可能不能完整支持它，所以本节全部用 CPU 跑，先把基础概念学扎实。

In [ ]:
import torch
import numpy as np

print("torch version:", torch.__version__)

# 先固定用 CPU，避免 CUDA 版本和新显卡架构不兼容导致初学阶段被环境问题打断。
device = torch.device("cpu")
print("device:", device)

## 2. Tensor 最重要的 5 个属性

以后调试模型，先看这 5 个属性：

| 属性/方法 | 人话解释 | 常见用途 |
|---|---|---|
| `x.shape` / `x.size()` | 张量形状 | 检查维度是否匹配 |
| `x.ndim` / `x.dim()` | 张量有几维 | 判断标量、向量、矩阵还是更高维 |
| `x.dtype` | 元素类型 | 特征通常 `float32`，分类标签常用 `long` |
| `x.device` | 张量在哪个设备上 | CPU/GPU 混用会报错 |
| `x.requires_grad` | 是否需要记录梯度 | 模型参数需要，普通标签不需要 |

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])

print("x =\n", x)
print("shape:", x.shape)
print("size():", x.size())
print("ndim:", x.ndim)
print("dtype:", x.dtype)
print("device:", x.device)
print("requires_grad:", x.requires_grad)

## 3. 创建 Tensor：从已有数据创建

### 3.1 `torch.tensor(data, dtype=None, device=None, requires_grad=False)`

作用：把 Python 数字、列表、嵌套列表、NumPy 数组转换成 Tensor。

常用参数：

| 参数 | 说明 |
|---|---|
| `data(原始数据)` | 比如数字、列表、NumPy 数组 |
| `dtype(元素类型)` | 指定元素类型，比如 `torch.float32`、`torch.int64` |
| `device(设备)` | 指定设备，比如 `"cpu"`、`"cuda"` |
| `requires_grad(是否追踪梯度)` | 训练模型参数时才常用 |

初学建议：优先用 `torch.tensor(...)`，清楚、稳定、可读性高。

In [ ]:
a = torch.tensor(10)
b = torch.tensor([1, 2, 3])
c = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
d = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

for name, value in {"a": a, "b": b, "c": c, "d": d}.items():
    print(name, value, "shape=", value.shape, "dtype=", value.dtype, "requires_grad=", value.requires_grad)

### 3.2 `torch.Tensor(...)`、`torch.FloatTensor(...)`、`torch.IntTensor(...)`

这些是早期常见写法，很多课程会讲，所以你要认识它们。

| 写法 | 作用 | 注意 |
|---|---|---|
| `torch.Tensor(data)` | 创建浮点 Tensor | 默认通常是 `float32` |
| `torch.Tensor(2, 3)` | 创建形状为 `[2, 3]` 的未初始化 Tensor | 值是内存里的旧值，不是 0 |
| `torch.FloatTensor(data)` | 创建 `float32` Tensor | 老写法，可读性一般 |
| `torch.IntTensor(data)` | 创建 `int32` Tensor | 分类标签一般更常用 `long/int64` |

初学建议：能看懂这些写法，但自己写新代码时优先用 `torch.tensor(..., dtype=...)`、`torch.zeros(...)`、`torch.ones(...)` 等更明确的函数。

In [ ]:
old_style_1 = torch.Tensor([1, 2, 3])
old_style_2 = torch.FloatTensor([1, 2, 3])
old_style_3 = torch.IntTensor([1, 2, 3])

print(old_style_1, old_style_1.dtype)
print(old_style_2, old_style_2.dtype)
print(old_style_3, old_style_3.dtype)

# 这个只分配空间，不保证里面是什么值。不要把它当全 0 张量。
uninitialized = torch.Tensor(2, 3)
print(uninitialized)

## 4. 创建固定值 Tensor

这类函数用于初始化数据、占位、构造 mask、构造标签等。

| 方法 | 作用 | 常用参数 | 例子 |
|---|---|---|---|
| `torch.zeros(size(形状))` | 创建全 0 Tensor | `dtype(元素类型)`, `device(设备)` | `torch.zeros(2, 3)` |
| `torch.ones(size(形状))` | 创建全 1 Tensor | `dtype(元素类型)`, `device(设备)` | `torch.ones(2, 3)` |
| `torch.full(size(形状), fill_value(填充值))` | 创建指定值 Tensor | `dtype(元素类型)` | `torch.full((2, 3), 7)` |
| `torch.empty(size(形状))` | 只分配空间，不初始化 | `dtype(元素类型)`, `device(设备)` | 很少给初学者直接用 |
| `torch.eye(n(行数), m(列数))` | 创建单位矩阵 | `dtype(元素类型)` | 线性代数里常见 |

注意：`size` 可以写成 `torch.zeros(2, 3)`，也可以写成 `torch.zeros((2, 3))`。

In [ ]:
print("zeros:\n", torch.zeros(2, 3))  # size(形状)
print("ones:\n", torch.ones(2, 3))  # size(形状)
print("full:\n", torch.full((2, 3), 7))  # size(形状), fill_value(填充值)
print("eye:\n", torch.eye(3))  # n(行数)

### 4.1 `xxx_like`：照着别人的形状创建

`zeros_like`、`ones_like`、`full_like` 的意思是：形状、设备、类型默认参考已有 Tensor。

这在写模型时很常用，因为你经常想创建一个“和输入同形状”的 mask、权重或临时变量。

In [ ]:
base = torch.tensor([[1.0, 2.0], [3.0, 4.0]])

print(torch.zeros_like(base))
print(torch.ones_like(base))
print(torch.full_like(base, 9.0))

## 5. 创建线性序列和随机 Tensor

| 方法 | 作用 | 常用参数                                 | 适合场景        |
|---|---|--------------------------------------|-------------|
| `torch.arange(start, end, step)` | 像 Python `range`，左闭右开 | `start(起始值)`, `end(终止值)`, `step(步长)` | 生成整数序列、索引   |
| `torch.linspace(start(起始值), end(终止值), steps(元素个数))` | 在区间内等间隔取点，包含两端 | `start(起始值)`, `end(终止值)`, `steps(分成的元素个数)`     | 画函数、造实验数据   |
| `torch.rand(size(形状))` | `[0, 1)` 均匀分布随机数 | `size(几行几列)`                         | 注意使用随机种子    |
| `torch.randn(size(形状))` | 标准正态分布随机数 N(0, 1) | `size(几行几列)`                               | 深度学习初始化、造噪声 |
| `torch.randint(low(下限), high(上限), size(形状))` | 随机整数，左闭右开 | `low(下限)`, `high(上限)`, `size(几行几列)`                | 随机类别、随机索引   |
| `torch.manual_seed(seed(随机种子值))` | 固定随机种子 | `seed(随机种子值)`                               | 让实验可复现      |

人话区别：`rand` 是 0 到 1 的随机小数；`randn` 是均值约 0、标准差约 1 的随机数，可能为负。

In [ ]:
torch.manual_seed(42)  # seed(随机种子值)

print("arange:", torch.arange(0, 10, 2))  # start(起始值), end(终止值), step(步长)
print("linspace:", torch.linspace(0, 1, 5))  # start(起始值), end(终止值), steps(元素个数)
print("rand:\n", torch.rand(2, 3))  # size(形状)
print("randn:\n", torch.randn(2, 3))  # size(形状)
print("randint:", torch.randint(0, 10, (5,)))  # low(下限), high(上限), size(形状)

## 6. 数据类型 dtype：别小看它

深度学习里最常见的 dtype：

| dtype | 人话解释 | 常见场景 |
|---|---|---|
| `torch.float32` / `torch.float` | 单精度浮点数 | 模型输入、权重、回归标签 |
| `torch.float64` / `torch.double` | 双精度浮点数 | 科学计算多，深度学习较少用 |
| `torch.int64` / `torch.long` | 64 位整数 | 多分类标签、索引 |
| `torch.int32` / `torch.int` | 32 位整数 | 一般整数数据 |
| `torch.bool` | 布尔值 | mask、条件筛选 |

两个非常常见的坑：

- 神经网络输入通常要是浮点数，不是整数。
- `nn.CrossEntropyLoss` 的分类标签通常要是 `torch.long`，不是 one-hot，也不是 float。

### 6.1 类型转换怎么写

| 写法 | 作用 | 人话说明 |
|---|---|---|
| `x.float()` | 转成 `torch.float32` | 神经网络输入最常用 |
| `x.double()` | 转成 `torch.float64` | 精度更高，但训练里不常用 |
| `x.long()` | 转成 `torch.int64` | 多分类标签常用 |
| `x.int()` | 转成 `torch.int32` | 普通整数 |
| `x.bool()` | 转成布尔类型 | mask、条件筛选 |
| `x.to(dtype=torch.float32)` | 用 `to` 指定目标类型 | 写法更统一，也能同时换 device |

注意：这些转换通常会返回一个新的 Tensor，不会原地修改旧 Tensor。也就是说，`x.float()` 只是得到一个 float 版本；如果你想让变量 `x` 以后都变成 float，要写 `x = x.float()`。

In [ ]:
x_int = torch.tensor([1, 2, 3])

x_float = x_int.float()      # int64 -> float32
x_long = x_float.long()      # float32 -> int64
x_double = x_float.double()  # float32 -> float64
x_bool = x_int.bool()        # 非 0 变 True，0 变 False

print("x_int dtype:", x_int.dtype)
print("x_float dtype:", x_float.dtype)
print("x_long dtype:", x_long.dtype)
print("x_double dtype:", x_double.dtype)
print("x_bool dtype:", x_bool.dtype)

# 也可以用 .to(dtype=...)
x_to_float = x_int.to(dtype=torch.float32)
print("x_to_float dtype:", x_to_float.dtype)

# 原来的 x_int 没有被改掉
print("x_int still dtype:", x_int.dtype)

## 7. Tensor 和 NumPy 互转

| 方法 | 作用 | 是否可能共享内存 | 什么时候用 |
|---|---|---:|---|
| `torch.from_numpy(arr)` | NumPy 转 Tensor | 是 | 想保留和 NumPy 的联系 |
| `torch.tensor(arr)` | NumPy/列表 转 Tensor | 否，通常复制数据 | 想要独立 Tensor |
| `torch.as_tensor(arr)` | 尽量不复制地转 Tensor | 可能 | 追求效率但要懂共享风险 |
| `tensor.numpy()` | Tensor 转 NumPy | CPU Tensor 上通常共享 | 画图、传统工具处理 |

人话提醒：如果你不想两个变量互相影响，就用 `.clone()` 或 `torch.tensor(...)` 明确复制。

In [ ]:
arr = np.array([1, 2, 3], dtype=np.float32)

t_shared = torch.from_numpy(arr)
t_copied = torch.tensor(arr)

arr[0] = 100

print("arr:", arr)
print("from_numpy shares memory:", t_shared)
print("torch.tensor copied data:", t_copied)